In [ ]:
# Testing Cell
import aviary.api as av
from aviary.utils.doctape import glue_variable, glue_keys, glue_class_methods, get_function_names


glue_keys(dict(av.ProblemType.__members__))
ProblemType = av.ProblemType
glue_variable(ProblemType.__name__, md_code=True)
glue_variable('Settings.PROBLEM_TYPE', av.Settings.PROBLEM_TYPE, md_code=True)

file_path = av.get_path('core/aviary_problem.py')
function_names = get_function_names(file_path)

for function_name in function_names:
    glue_variable(function_name + '()', md_code=True)

# glue design variables
glue_variable('Mission.GROSS_MASS', av.Mission.GROSS_MASS, md_code=True)
glue_variable('Aircraft.Design.GROSS_MASS', av.Aircraft.Design.GROSS_MASS, md_code=True)
glue_variable('Mission.RANGE', av.Mission.RANGE, md_code=True)

# glue objectives
glue_variable('Mission.Objectives.FUEL', av.Mission.Objectives.FUEL, md_code=True)
glue_variable('Mission.Objectives.RANGE', av.Mission.Objectives.RANGE, md_code=True)

# glue constraints
glue_variable(
    'Mission.Constraints.RANGE_RESIDUAL', av.Mission.Constraints.RANGE_RESIDUAL, md_code=True
)
glue_variable(
    'Mission.Constraints.MASS_RESIDUAL', av.Mission.Constraints.MASS_RESIDUAL, md_code=True
)

str_problem_type = av.Settings.PROBLEM_TYPE
str_off_design_min_fuel = av.ProblemType.OFF_DESIGN_MIN_FUEL.value

str_off_design_min_fuel_snippet = f'```\n{str_problem_type},{str_off_design_min_fuel},unitless\n```'
glue_variable('off_design_min_fuel_csv', str_off_design_min_fuel_snippet, md_code=False)

file_path = av.get_path('docs/examples/off_design_missions.ipynb').relative_to(av.top_dir.parent)
glue_variable('off_design_missions', file_path, md_code=True)


glue_variable(
    'Aircraft.CrewPayload.CARGO_CONTAINER_MASS',
    av.Aircraft.CrewPayload.CARGO_CONTAINER_MASS,
    md_code=True,
)
glue_variable(
    'Aircraft.CrewPayload.NUM_FIRST_CLASS',
    av.Aircraft.CrewPayload.NUM_FIRST_CLASS,
    md_code=True,
)

glue_variable('ENERGY_STATE', av.EquationsOfMotion.ENERGY_STATE.name, md_code=True)
glue_variable(
    'TWO_DEGREES_OF_FREEDOM', av.EquationsOfMotion.TWO_DEGREES_OF_FREEDOM.name, md_code=True
)

# Off-Design Missions

## Overview

Off-design missions are missions that take an already designed and sized aircraft and attempt to run different mission trajectories, different payload quantities, or both.

There are two types of off-design missions supported in Aviary:

- {glue:md}`OFF_DESIGN_MIN_FUEL` Missions: the mission's target range and aircraft payload mass are inputs and the fuel mass required is solved for. 
- {glue:md}`OFF_DESIGN_MAX_RANGE` Missions: the aircraft payload and gross mass are inputs and the range of the aircraft is solved for.

There are 2 methods of running off-design missions:
1. Modifying {glue:md}`Settings.PROBLEM_TYPE` variable in an aviary input .csv or aviary values object.
2. Calling {glue:md}`run_off_design_mission()` after sizing an aircraft using the python api.

## Running Off-Design Using the {glue:md}`Settings.PROBLEM_TYPE` Variable:

The {glue:md}`Settings.PROBLEM_TYPE` variable provides a convenient way for the user to set common combinations of design variables, constraints and objective function for solving aircraft design or performance evaluation problems.

- {glue:md}`SIZING` Missions allow the optimizer to control both {glue:md}`Aircraft.Design.GROSS_MASS` and {glue:md}`Mission.GROSS_MASS` for the given mission and objective. 
- {glue:md}`OFF_DESIGN_MIN_FUEL` Missions allow the optimizer to only control the {glue:md}`Mission.GROSS_MASS` to converge for a specified target range. {glue:md}`Aircraft.Design.GROSS_MASS` is fixed for an already sized aircraft. 
- {glue:md}`OFF_DESIGN_MAX_RANGE` Missions allow the optimizer to maximize {glue:md}`Mission.RANGE` for a given input {glue:md}`Mission.GROSS_MASS`. {glue:md}`Aircraft.Design.GROSS_MASS` is fixed for an already sized aircraft.
- {glue:md}`MULTI_MISSION` typically allow the optimizer to control {glue:md}`Aircraft.Design.GROSS_MASS` and the {glue:md}`Mission.GROSS_MASS` for each of the missions within multimission. See [multi_mission documentation](../user_guide_unreviewed/multi_mission.ipynb) for full details.

This table contains a summary of the preset combinations of objectives, design variables and constraints.

| {glue:md}`Settings.PROBLEM_TYPE` | objective | design variables | constraints |
| ------------------------| --------- | ---------------- | ----------- |
| {glue:md}`SIZING`       | {glue:md}`Mission.Objectives.FUEL` | {glue:md}`Mission.GROSS_MASS`, {glue:md}`Aircraft.Design.GROSS_MASS` | {glue:md}`Mission.Constraints.RANGE_RESIDUAL`, {glue:md}`Mission.Constraints.MASS_RESIDUAL` |
| {glue:md}`OFF_DESIGN_MIN_FUEL` | {glue:md}`Mission.Objectives.FUEL`| {glue:md}`Mission.GROSS_MASS` | {glue:md}`Mission.Constraints.RANGE_RESIDUAL` |
| {glue:md}`OFF_DESIGN_MAX_RANGE` | {glue:md}`Mission.Objectives.RANGE` | No additional design variables | No additional constraints |
| {glue:md}`MULTI_MISSION` | No default, user specified | Custom setup - see multimission documentation | Custom setup - see multimission documentation |

```{note}
This table does not show all the design variables for the problem, only those altered by the {glue:md}`Settings.PROBLEM_TYPE` variable.
Dymos will add additional design variables and constraints depending on number of phases, whether mach and altitude are to be optimized during those phases and any additional constraints in each phase.
For {glue:md}`TWO_DEGREES_OF_FREEDOM` equations of motion, there are additional design variables due to increased fidelity of the trajectory modelling.
```

The {glue:md}`Settings.PROBLEM_TYPE` variable can be set for off design missions in the aviary input .csv file:

> {glue:md}`off_design_min_fuel_csv`

If manually creating an aviary values object then it can be set before calling {glue:md}`load_inputs()`:

> from aviary.variable_info.enums import ProblemType

> aviary_values.set_val(Settings.PROBLEM_TYPE, ProblemType.OFF_DESIGN_MAX_RANGE)

```{note}
{glue:md}`MULTI_MISSION` problem type must be set earlier when initializing the Aviary problem object itself, see multi mission documentation for more details.
```

Run the rest of the aviary problem as normal from this point and Aviary will take care of adding the appropriate objective, design variables, and constraints as detailed in the table above.

```{note}
Since Aviary is not re-sizing the aircraft it is important that all the variables in the aviary_values or .csv input contian the appropriate values to define an already sized aircraft.
```

## Running Off-Design using the Python API {glue:md}`run_off_design_mission()` method:

Following the successful convergence of a {glue:md}`SIZING` type aircraft and misison optimization, the {glue:md}`run_off_design_mission()` method can be run from the same python script.

```{note}
If the {glue:md}`SIZING` mission did not converge to a valid aircraft design, any off-design analysis will be invalid even if the off-design missions themselves converged.
```

This method has a number of arguments that tell aviary the conditions required for the off_design_mission. A simple example is shown in the [off_design_mission example]({glue:md}`off_design_missions`)
If any of the arguments are unspecified or None then Aviary will assume the same values as for the {glue:md}`SIZING` problem. Full details of all the arguments can be found in the [source docs](../_srcdocs/packages/core/aviary_problem.md).
The key arguments allow the user to set the problem_type, phase_info, payload (passengers + cargo) and fuel loading or mission range depending on the problem type.
There are 2 additional flags that allow the user to add aditional design variables to maximise gross mass or maximise cargo mass for a particular off-design mission.

The `phase_info` argument allows the user to specify different phase information for an off-design mission trajectory, more information can be found in [The `phase_info` Format](../source_docs/phase_info_detailed.ipynb).
This can be required to achieve convergence when the off-design mission has a significantly different trajectory shape or duration compared to the the sizing mission. 
For example, an aircraft sized for a very long-range mission might need its climb and descent profiles modified to operate on a shorter route.

```{note}
Off-design missions are run with the assumption that the sizing mission's {glue:md}`Aircraft.Design.GROSS_MASS` is the maximum structural mass the aircraft can support. 
However, since {glue:md}`Mission.Constraints.MASS_RESIDUAL` is not added as a constraint for off-design, Aviary is capable of running and converging off-design missions with {glue:md}`Mission.GROSS_MASS` values greater than {glue:md}`Aircraft.Design.GROSS_MASS`. The user is responsible for checking that they have run valid off-design missions.
```

## Saving and Loading the sized aircraft

The {glue:md}`save_results()` method can be used to save the Aviary problem to a JSON file so that it can be loaded and used for off-design mission analysis.

> `prob.save_results(json_filename = 'sizing_problem.json')` 

```{note}
This doesn't actually save all the values in the Aviary problem, but saves the aviary_inputs and value of the Aircraft.Design.GROSS_MASS variable (which is all that is required for an off_design_mission).
The Aviary team is working on a more complete method of saving the design aircraft details.
``` 

The {glue:md}`_read_sizing_json()` method can be used to create an aviary values object from the data saved into a JSON file using the {glue:md}`save_results()` method.

> `aviary_values = _read_sizing_json(json_filename = 'sizing_problem.json')`

This aviary values object can then be fed into an aviary problem using the {glue:md}`load_inputs()` method.
The off-design mission analysis can then be run using method 1 above.

```{note} 
As of v0.9.10 Aviary's off-design capabilities do not allow missions to be run with 0 passengers, so a minimum of 1 passenger must be specified. 
This is a known bug and is currently being investigated. 
```

```{note} 
Off-design missions **cannot** be run with more passengers than the original sizing mission. 
For example, if {glue:md}`Aircraft.CrewPayload.NUM_FIRST_CLASS` within the Aviary inputs csv was set to 3, an off-design mission cannot be run where {glue:md}`Aircraft.CrewPayload.NUM_FIRST_CLASS` = 8 as the cabin for first class was sized for precisely 3 passengers.
```

```{note}
In cases where passengers or cargo are changed and the user has not specified a {glue:md}`Aircraft.CrewPayload.CARGO_CONTAINER_MASS` in the Aviary input deck, this value is recalculated in the off-design mission.
It is the only parameter within the operating mass that gets recalculated between off-design and sizing. If this behavior is undesirable then a fixed override for {glue:md}`Aircraft.CrewPayload.CARGO_CONTAINER_MASS` should be used.
```

## Payload Range Diagram Functionality

The Off-Design functionality is key for the generation of Payload Range Diagrams. See [payload range documentation](../user_guide_unreviewed/payload_range_functionality.ipynb) for more details.

##